Web Crawler data flow: 

1. Take seed URL from frontier and request IP from DNS
2. Fetch HTML from external server using IP
3. Extract text data from the HTML.
4. Store the text data in a database.
5. Extract any linked URLs from the web pages and add them to the list (Frontier Queue) of URLs to crawl.
6. Repeat steps 1-5 until all URLs have been crawled.

In [ ]:
TESTING_SEED_URL = "https://softwarica.edu.np/courses"
TESTING_ALLOWED_DOMAIN = "softwarica.edu.np"

TESTING_CRAWL_DELAY_SECONDS = 5*24*60*60      # 5 days
TESTING_MAX_PAGES = 100               
TESTING_USER_AGENT = "SoftwaricaVerticalSearchBot/1.0 (+educational IR project)"

In [2]:
MAIN_SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/"
MAIN_ALLOWED_DOMAIN = "pureportal.coventry.ac.uk"
MAIN_CRAWL_DELAY_SECONDS = 24*60*60*30*3    # 3 months      
MAIN_MAX_PAGES = 100               
MAIN_USER_AGENT = "CoventryVerticalSearchBot/1.0 (+educational IR project)"

Ensuring Politness steps: 

1. To make this clear, the steps would be:
2. Fetch the `robots.txt` file for the domain.
3. Parse the `robots.txt` file and store it in the database (MongoDB).
4. When we pull a URL off the queue, check the rules stored in the database (MongoDB) for that domain.
5. If the URL is disallowed, ack the message and move on to the next URL.
6. If the URL is allowed, check the `Crawl-delay` directive.

IF Crawler is failed

7. If the Crawl-delay time has not passed since the last crawl, use ChangeMessageVisibility to extend the visibility timeout and defer reprocessing.
8. If the Crawl-delay time has passed, crawl the page and update the last crawl time for the domain.

In [3]:
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests

In [12]:
base_url = TESTING_SEED_URL
USER_AGENT = TESTING_USER_AGENT


In [5]:
# fetching the content of the robots.txt file 

def fetch_robots(base_url, USER_AGENT):
    parsed_url = urlparse(base_url)     # Example: ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')
    robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
    rfp = RobotFileParser()
    rfp.set_url(robots_url)

    try: 
        res = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=10)
        if res.status_code == 200: 
            rfp.parse(res.text.splitlines())
        else: 
            rfp = None
    except BaseException as err: 
        print(f"Error: {err}")
        rfp = None
        
    return rfp

In [6]:
robots_data = fetch_robots(TESTING_SEED_URL, TESTING_USER_AGENT)
print(robots_data)

User-agent: Amazonbot
Disallow: /

User-agent: Applebot-Extended
Disallow: /

User-agent: Bytespider
Disallow: /

User-agent: CCBot
Disallow: /

User-agent: ClaudeBot
Disallow: /

User-agent: CloudflareBrowserRenderingCrawler
Disallow: /

User-agent: Google-Extended
Disallow: /

User-agent: GPTBot
Disallow: /

User-agent: meta-externalagent
Disallow: /

User-agent: *
Allow: /


In [8]:
def can_fetch(rfp, url):
    if rfp is None: 
        return True
    return rfp.can_fetch(USER_AGENT, url)

In [9]:
can_fetch(robots_data, TESTING_SEED_URL)

True

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
# extracting HTML content (links, title, text, etc)

# def extract_content(html, base_url):
res = requests.get(TESTING_SEED_URL)
if res != 200:
    print(f"Extracting HTML content failed")

html_content = res.text
soup = BeautifulSoup(html_content, 'html.parser')

# extracted links
extracted_links = set()

for a in soup.find_all("a", href=True):
    # getting each link 
    link = urljoin(base_url, a["href"]).split("#")[0]
    parsed = urlparse(link)
    if parsed.netloc.endswith(TESTING_ALLOWED_DOMAIN) and parsed.scheme in ("https", "http"):
        extracted_links.add(link)
        
print(extracted_links)

Extracting HTML content failed
<a class="flex items-center" href="/"><img alt="Softwarica Logo" class="h-20 w-auto" data-nimg="1" decoding="async" height="80" src="/_next/image?url=%2Flogo.png&amp;w=640&amp;q=75" srcset="/_next/image?url=%2Flogo.png&amp;w=384&amp;q=75 1x, /_next/image?url=%2Flogo.png&amp;w=640&amp;q=75 2x" style="color:transparent" width="263"/></a>
<a class="transition-colors duration-200 hover:text-[#1b6a9c] uppercase text-sm font-medium text-gray-800" href="/courses">Course</a>
<a class="transition-colors duration-200 hover:text-[#1b6a9c] uppercase text-sm font-medium text-gray-800" href="/contact">Contact Us</a>
<a class="transition-colors duration-200 hover:text-[#1b6a9c] uppercase text-sm font-medium text-gray-800" href="/student-center/notices">Notices</a>
<a class="px-4 md:px-6 py-2 md:py-2.5 rounded-lg transition-all duration-200 font-medium text-xs md:text-sm mr-2 md:mr-3 bg-[#1b6a9c] hover:bg-[#145580] text-white hover:shadow-lg" href="https://c4mpus.com" re

In [16]:
# decomposing the unwanted tags
for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
    tag.decompose()

In [21]:
import re

In [22]:
# extracting title, text
if soup.title and soup.title.string:
    title = soup.title.string.strip()

else:
    title = ""
    
print(title)

text = soup.get_text(separator=" ")
text = re.sub(r"\s+", " ", text).strip()
print(text)

Programs & Courses | Softwarica College
Programs & Courses | Softwarica College Loading Our Programs Explore our range of internationally recognized degree programs Filters All Levels Bachelors' Masters' Featured Programs


Note: Rate limiting (avoiding system crash by requesting to crawl) is important. Sliding window algorithm can be used to track the number of requests per domain per second